# Automatic detection of text areas

In [1]:
import cv2
import numpy as np

def detect_code_regions(image):
    """
    image: input page scan (BGR or grayscale)
    returns: list of bounding boxes [(x, y, w, h)]
    """

    # -------------------------
    # 1. Preprocessing
    # -------------------------

    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()

    # Normalize lighting a bit
    gray = cv2.GaussianBlur(gray, (25, 15), 0)

    # -------------------------
    # 2. Binarization
    # -------------------------

    binary = cv2.adaptiveThreshold(
        gray,
        maxValue=255,
        adaptiveMethod=cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        thresholdType=cv2.THRESH_BINARY_INV,
        blockSize=45,
        C=10
    )

    # -------------------------
    # 3. Morphological grouping
    # -------------------------

    # This kernel joins handwriting strokes horizontally
    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (100, 20)
        #(25, 5)   # width >> height
    )

    grouped = cv2.morphologyEx(
        binary,
        cv2.MORPH_CLOSE,
        kernel
    )

    # -------------------------
    # 4. Connected components
    # -------------------------

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        grouped,
        connectivity=8
    )

    candidates = []

    h_img, w_img = gray.shape

    for i in range(1, num_labels):  # skip background
        x, y, w, h, area = stats[i]

        # -------------------------
        # 5. Geometric filtering
        # -------------------------

        aspect_ratio = w / float(h)
        fill_ratio = area / float(w * h)

        # Typical handwritten 5-letter code heuristics
        if area < 1000:          # too small
            continue
        #if area > 20000:        # too big
        #    continue
        if aspect_ratio < 1.0:   # too narrow
            continue
        if aspect_ratio > 15.0:  # too wide
            continue
        if fill_ratio < 0.25:    # too sparse
            continue

        # Optional: restrict search area
        if x < w_img * 0.4:     # ignore top part of page
            continue

        candidates.append((x, y, w, h, area))

    # Sort candidates by likelihood (example: area descending)
    candidates = sorted(
        candidates,
        key=lambda b: b[2] * b[3],
        reverse=True
    )

    # Return top N candidates
    return candidates[:5]


In [2]:
from zettelsortierung.Zettelwerk import Zettel, Zettelsammlung
import os
from dotenv import load_dotenv

load_dotenv()

root = os.getenv('ZETTELSAMMLUNG_ROOT')
sammlung = Zettelsammlung.collect_zettel(root, 125)
sammlung = sammlung[75:100]
print(len(sammlung))

for idx, zettel in enumerate(sammlung):
    img = cv2.imread(zettel.recto_file_path, 0)
    boxes = detect_code_regions(img)

    for jdx, box in enumerate(boxes):
        x, y, w, h, area = box
        #area = w * h
        aspect_ratio = w / float(h)
        fill_ratio = area / float(w * h)

        stats = f'Box {jdx}: {area}, {aspect_ratio:.2f}, {fill_ratio:.2f}'

        img = cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 0), 10)
        img = cv2.putText(img, stats, (x-60, y-10), 0, 1.0, (0, 0, 0), 2)
    img = cv2.resize(img, (1500, 1000))
    cv2.imshow("image", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


17it [00:00, 59.65it/s]


25


In [ ]:
from zettelsortierung import WritingDetection
from zettelsortierung.Zettelwerk import Zettel, Zettelsammlung
import os
import cv2
from dotenv import load_dotenv

load_dotenv()

root = os.getenv('ZETTELSAMMLUNG_ROOT')
sammlung = Zettelsammlung.collect_zettel(root, 125)
sammlung = sammlung[99:100]
print(len(sammlung))

for idx, zettel in enumerate(sammlung):
    img = cv2.imread(zettel.recto_file_path, 0)
    boxes = WritingDetection.detect_code_regions(img)
    WritingDetection.display_boxes(img, boxes)


17it [00:00, 54.54it/s]


1
<class 'tuple'>
